In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install python-Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.4/177.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 52.0 MB/s eta 0:00:00


In [ ]:
import os
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from scipy.spatial.transform import Slerp, Rotation
from scipy.interpolate import interp1d
import torch.nn.functional as F

class HandGestureDataset(Dataset):
    def __init__(self, root_dir, num_classes_per_batch, num_distractions_per_batch, max_sequence_length, batch_length=100, mirror = True, speed_variation=0.5, random_slice=True, sample_all=False, train=True):
        self.root_dir = root_dir
        self.num_classes_per_batch = num_classes_per_batch
        self.num_distractions_per_batch = num_distractions_per_batch
        self.max_sequence_length = max_sequence_length
        self.speed_variation = speed_variation
        self.random_slice = random_slice
        self.batch_length = batch_length
        self.sample_all = sample_all
        self.train = train
        self.mirror = mirror

        self.class_names = sorted(os.listdir(root_dir))
        self.class_to_idx = {}
        self.data_files = {}
        for i, class_name in enumerate(self.class_names):
            class_dir = os.path.join(root_dir, class_name)
            self.data_files[class_name] = sorted(os.listdir(class_dir))
            self.class_to_idx[class_name] = i

        # Split the data files into training and validation sets
        self.train_files = {}
        self.val_files = {}
        for class_name, files in self.data_files.items():
            split_idx = int(0.8 * len(files))
            self.train_files[class_name] = files[:split_idx]
            self.val_files[class_name] = files[split_idx:]

    def __len__(self):
        return self.batch_length

    def vary_speed(self, gesture_data):
        # Randomly choose a speed factor
        speed_factor = np.random.uniform(1 - self.speed_variation, 1 + self.speed_variation)

        # Calculate new number of frames
        original_frames = gesture_data.shape[0]
        new_frames = int(original_frames / speed_factor)

        # Create interpolation function for each dimension
        old_times = np.arange(original_frames)
        new_times = np.linspace(0, original_frames - 1, new_frames)

        interpolator = interp1d(old_times, gesture_data, axis=0, kind='linear')

        # Interpolate to get new gesture data
        new_gesture_data = interpolator(new_times)

        return new_gesture_data

    def vary_position(self, gesture_data):
      return gesture_data + np.random.uniform(-0.05, 0.05, (3,))

    def concatenate_hand_gestures(self, gesture_data_list, class_indices):
        concatenated_data = []
        concatenated_labels = []
        transition_frames_length = np.random.randint(5, 30, size=len(gesture_data_list)-1)

        for i, (current_data, class_idx) in enumerate(zip(gesture_data_list, class_indices)):
            # Append the current gesture data and its label
            concatenated_data.append(current_data)
            concatenated_labels.extend([class_idx] * len(current_data))

            if i < len(gesture_data_list) - 1:
                next_data = gesture_data_list[i + 1]

                # Get the end frame of the current gesture and the start frame of the next gesture
                end_frame = current_data[-1]
                start_frame = next_data[0]

                # Create a list to store the interpolated frames
                transition_data = []

                # Interpolate each landmark separately
                for j in range(42):
                    # Create a Rotation object for the current landmark
                    rotations = Rotation.from_rotvec(np.vstack([end_frame[j], start_frame[j]]))

                    # Create a Slerp object for interpolation
                    slerp = Slerp(times=[0, 1], rotations=rotations)

                    # Generate interpolated frames for the transition
                    transition_times = np.linspace(0, 1, transition_frames_length[i])
                    landmark_transition = slerp(transition_times).as_rotvec()

                    # Append the interpolated landmark frames to the transition data
                    transition_data.append(landmark_transition)

                # Convert the transition data to a NumPy array with shape (transition_frames, 42, 3)
                transition_data = np.transpose(np.array(transition_data), (1, 0, 2))

                # Append the transition frames to the concatenated data
                concatenated_data.append(transition_data)
                concatenated_labels.extend([-1] * transition_frames_length[i])

        # Stack the concatenated data into a single array
        concatenated_data = np.vstack(concatenated_data)
        concatenated_labels = np.array(concatenated_labels)

        return concatenated_data, concatenated_labels

    def __getitem__(self, index):
        # Use different file sets based on whether this is training or validation
        file_set = self.train_files if self.train else self.val_files

        # Randomly select classes for the input sequence
        input_classes = random.sample(self.class_names, self.num_classes_per_batch)

        # Load the input sequences
        gesture_data_list = []
        class_indices = []
        for class_name in input_classes:
            data_file = random.choice(file_set[class_name])
            data_path = os.path.join(self.root_dir, class_name, data_file)
            gesture_data = np.load(data_path)
            gesture_data = self.vary_speed(self.vary_position(gesture_data))
            if self.mirror:
              gesture_data[:,:,0] *= -1
              gesture_data[:,:,0] += 1
            gesture_data_list.append(gesture_data)
            class_indices.append(self.class_to_idx[class_name])

        # Concatenate the gestures with SLERP interpolation
        input_sequence, input_label = self.concatenate_hand_gestures(gesture_data_list, class_indices)

        # Randomly slice the input sequence and label if longer than max_sequence_length
        if self.random_slice:
            start_idx = random.randint(0, min(15, len(input_sequence)//2))
            end_idx = len(input_sequence)-random.randint(0, min(15, len(input_sequence)//2))
            input_sequence = input_sequence[start_idx:end_idx]
            curr_idx = start_idx
            prev=input_label[start_idx]
            while input_label[curr_idx] == prev:
              input_label[curr_idx] = -2 #-2 represents sliced inputs. The model will ignore these
              curr_idx += 1

            curr_idx = end_idx
            prev=input_label[end_idx - 1]
            while input_label[curr_idx - 1] == prev:
              input_label[curr_idx - 1] = -2
              curr_idx -= 1

            input_label = input_label[start_idx:end_idx]


        # Convert to PyTorch tensors
        input_sequence = torch.from_numpy(input_sequence).float()
        input_label = torch.from_numpy(input_label).long()

        # Randomly select classes for sample sequences
        # input_classes = random.sample(input_classes, len(input_classes) - random.randint(0, 1))
        additional_classes = random.choices(self.class_names, k=self.num_distractions_per_batch)
        sample_classes = list(set(input_classes + additional_classes))
        if self.sample_all:
          sample_classes = self.class_names

        # Load all sample sequences
        sample_sequences = []
        sample_labels = []
        for class_name in sample_classes:
            data_file = random.choice(file_set[class_name])
            data_path = os.path.join(self.root_dir, class_name, data_file)
            gesture_data = np.load(data_path)
            gesture_data = self.vary_speed(self.vary_position(gesture_data))
            if self.mirror:
              gesture_data[:,:,0] *= -1
              gesture_data[:,:,0] += 1
            sample_sequences.append(torch.from_numpy(gesture_data).float())
            sample_labels.append(self.class_to_idx[class_name])

        # Find the maximum time length
        max_length = max(seq.shape[0] for seq in sample_sequences)

        # Pad sequences to max_length
        padded_sequences = []
        sample_masks = []
        for seq in sample_sequences:
            padding_length = max_length - seq.shape[0]
            padded_seq = F.pad(seq.permute(1, 2, 0), (0, padding_length), mode='constant', value=0)
            padded_sequences.append(padded_seq.permute(2, 0, 1))

            mask = torch.ones(max_length, dtype=torch.bool)
            mask[seq.shape[0]:] = 0
            sample_masks.append(mask)

        # Stack sequences, labels, and masks
        sample_sequences = torch.stack(padded_sequences)
        sample_labels = torch.tensor(sample_labels)
        sample_masks = torch.stack(sample_masks)

        return input_sequence, input_label, sample_sequences, sample_labels, sample_masks

random.seed(42)
np.random.seed(42)
torch.random.manual_seed(42)


root_dir = '/content/drive/MyDrive/SignData/test'

train_dataset = HandGestureDataset(root_dir, 10, 30, 150, train=True)
val_dataset = HandGestureDataset(root_dir, 10, 20, 150, sample_all=True, batch_length=10, train=False)


train_loader = DataLoader(train_dataset, shuffle=True, num_workers=2, batch_size=1)
val_loader = DataLoader(val_dataset, shuffle=False, num_workers=2, batch_size=1)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

class spatialExtractor(nn.Module):
    def __init__(self, input_channels, hidden_channels, output_channels):
        super(spatialExtractor, self).__init__()
        self.conv1 = nn.Conv2d(input_channels, hidden_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(hidden_channels)
        self.conv2 = nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(hidden_channels)
        self.conv3 = nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(hidden_channels)
        self.conv4 = nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(hidden_channels)
        self.conv5 = nn.Conv2d(hidden_channels, output_channels, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(output_channels)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = F.relu(self.bn5(self.conv5(x)))
        return x

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding):
        super(TemporalBlock, self).__init__()
        self.conv1 = nn.Conv1d(n_inputs, n_outputs, kernel_size,
                               stride=stride, padding=padding, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(n_outputs)
        self.relu1 = nn.ReLU()
        self.conv2 = nn.Conv1d(n_outputs, n_outputs, kernel_size,
                               stride=stride, padding=padding, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(n_outputs)
        self.relu2 = nn.ReLU()

        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
        self.init_weights()

    def init_weights(self):
        self.conv1.weight.data.normal_(0, 0.01)
        self.conv2.weight.data.normal_(0, 0.01)
        if self.downsample is not None:
            self.downsample.weight.data.normal_(0, 0.01)

    def forward(self, x):
        out = self.relu1(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TCNSignEmbedding(nn.Module):
    def __init__(self, embedding_dim=256, num_landmarks=42, input_channels=3, input_frames=30):
        super(TCNSignEmbedding, self).__init__()

        # Spatial feature extractor
        self.spatial_extractor = spatialExtractor(input_channels, 16, 32)

        # Dimensionality reduction
        self.dim_reduction = nn.Conv1d(32 * num_landmarks, embedding_dim, kernel_size=1)

        # Temporal modeling with TCN
        num_channels = [embedding_dim, embedding_dim, embedding_dim]
        self.tcn = nn.Sequential(
            TemporalBlock(num_channels[0], num_channels[1], kernel_size=3, stride=1, dilation=1, padding=1),
            TemporalBlock(num_channels[0], num_channels[1], kernel_size=3, stride=1, dilation=1, padding=1),
            TemporalBlock(num_channels[0], num_channels[1], kernel_size=3, stride=1, dilation=1, padding=1),
            TemporalBlock(num_channels[1], num_channels[2], kernel_size=3, stride=1, dilation=2, padding=2),
        )

        self.threshold = nn.Parameter(torch.tensor(4.0))


    def forward(self, x):
        # x shape: (batch_size, frames, landmarks, channels)
        batch_size, frames, landmarks, channels = x.size()

        # Reshape x to (batch_size * frames, channels, landmarks, 1)
        x = x.permute(0, 1, 3, 2).contiguous().view(-1, channels, landmarks, 1)

        # Apply CNN
        x = self.spatial_extractor(x)

        # Reshape back to (batch_size, frames, landmarks, features)
        _, features, _, _ = x.size()
        x = x.view(batch_size, frames, features, landmarks).permute(0, 2, 3, 1)

        # Prepare for dimensionality reduction (batch_size, features, frames)
        x = x.contiguous().view(batch_size, -1, frames)

        # Apply dimensionality reduction
        x = self.dim_reduction(x)

        # Temporal modeling
        x = self.tcn(x)


        # Reshape back to (batch_size, frames, features)
        x = x.permute(0, 2, 1).contiguous()

        return x


In [ ]:
from typing import List, Tuple
from numba import njit
import Levenshtein

@njit
def dtw_path(x: np.ndarray, y: np.ndarray) -> List[Tuple[int, int]]:
    N, E = x.shape
    M, _ = y.shape

    # Initialize the cost matrix with infinity
    dtw = np.full((N + 1, M + 1), np.inf)
    dtw[0, 0] = 0.0

    # Compute the cumulative distance and store the path
    for i in range(1, N + 1):
        for j in range(1, M + 1):
            cost = np.linalg.norm(x[i-1] - y[j-1])  # Euclidean distance between points
            dtw[i, j] = cost + min(dtw[i-1, j],    # Insertion
                                   dtw[i, j-1],    # Deletion
                                   dtw[i-1, j-1])  # Match

    # Backtrack to find the optimal path
    path = []
    i, j = N, M
    while i > 0 or j > 0:
        path.append((i-1, j-1))
        if i == 0:
            j -= 1
        elif j == 0:
            i -= 1
        else:
            # Choose the direction with the minimum accumulated cost
            min_cost_index = np.argmin(np.array([dtw[i-1, j], dtw[i, j-1], dtw[i-1, j-1]]))
            if min_cost_index == 0:
                i -= 1
            elif min_cost_index == 1:
                j -= 1
            else:
                i -= 1
                j -= 1

    # Reverse path to start from (0, 0)
    path.reverse()

    return path


def partition_sequence(sequence, labels):
  partitioned = []
  current_label = labels[0]
  current_partition = []

  for i, label in enumerate(labels):
      if label == current_label:
          current_partition.append(sequence[i])
      else:
          partitioned.append((torch.stack(current_partition), current_label))
          current_label = label
          current_partition = [sequence[i]]
  partitioned.append((torch.stack(current_partition), current_label))

  return partitioned

def dtw_partition_loss(input_embedding, target_embeddings, labels, threshold, alpha=0.1):
  episodes = partition_sequence(input_embedding, labels)
  total_predictions = 0
  correct_predictions = 0
  loss = 0
  for episode, label in episodes:
      if label == -2:
        continue
      distance = []
      discriminant = None
      for target_label, target_embedding in enumerate(target_embeddings):
        path = dtw_path(episode.detach().cpu().numpy(), target_embedding.detach().cpu().numpy())
        dtw_dist = torch.tensor(0.0, device=input_embedding.device)
        for (i, j) in path:
            dtw_dist += torch.norm(episode[i] - target_embedding[j])
        distance.append(dtw_dist)
        if target_label == label.item():
          discriminant = dtw_dist
      distance.append(threshold)
      output = -torch.stack(distance)
      loss += F.cross_entropy(output, label)
      if label!=len(target_embeddings):
        loss += alpha * discriminant


      predicted = torch.argmax(output)
      if predicted == label:
        correct_predictions += 1

      total_predictions += 1



  return loss, correct_predictions, total_predictions





@njit
def partial_DTW_helper(x: np.ndarray, y: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    N, E = x.shape
    M, _ = y.shape

    dtw = np.empty((N, M))
    origins = np.empty((N, M), dtype=np.int64)

    for i in range(N):
        dist = np.linalg.norm(x[i] - y[0])
        dtw[i, 0] = dist
        origins[i, 0] = i
    for j in range(1, M):
        dist = np.linalg.norm(x[0] - y[j])
        dtw[0, j] = dist + dtw[0, j-1]
        origins[0, j] = 0

    for i in range(1, N):
        for j in range(1, M):
            dist = np.linalg.norm(x[i] - y[j])

            # Manually determine the minimum cost path
            cost_diag = dtw[i-1, j-1]
            cost_left = dtw[i, j-1]
            cost_up = dtw[i-1, j]

            if cost_diag <= cost_left and cost_diag <= cost_up:
                dtw[i, j] = dist + cost_diag
                origins[i, j] = origins[i-1, j-1]
            elif cost_left < cost_up:
                dtw[i, j] = dist + cost_left
                origins[i, j] = origins[i, j-1]
            else:
                dtw[i, j] = dist + cost_up
                origins[i, j] = origins[i-1, j]

    return dtw, origins

@njit
def calculate_path(dtw: np.ndarray, start: int, end: int) -> List[Tuple[int, int]]:
  path = []
  N, M = dtw.shape
  i, j = end, M-1
  while i > start or j > 0:
      path.append((i, j))
      if i == start:
          j -= 1
      elif j == 0:
          i -= 1
      else:
          min_cost_index = np.argmin(np.array([dtw[i-1, j], dtw[i, j-1], dtw[i-1, j-1]]))
          if min_cost_index == 0:
              i -= 1
          elif min_cost_index == 1:
              j -= 1
          else:
              i -= 1
              j -= 1
  path.append((start, 0))
  return path


def partial_DTW_torch(x, y):
    dtw, origins = partial_DTW_helper(x.detach().cpu().numpy(), y.detach().cpu().numpy())

    curr_min_cost = torch.tensor(float('inf'))
    costs = torch.empty((len(x),), device=x.device)
    curr_origin = origins[-1, -1]

    for i in range(len(x)-1, -1, -1):
        if curr_min_cost.detach().item() + (dtw[i,0] if i < curr_origin else 0) > dtw[i, -1]:
            curr_origin = origins[i, -1]
            path = calculate_path(dtw, origins[i, -1], i)
            dtw_dist = 0
            for (I, J) in path:
                dtw_dist += torch.norm(x[I] - y[J])
            curr_min_cost = dtw_dist
            costs[i] = dtw_dist
        elif i < curr_origin:
          curr_min_cost += torch.norm(x[i] - y[0])

        costs[i] = curr_min_cost

    return costs



def partial_dtw_loss(input_embedding, target_embeddings, target_masks, labels, threshold):
  costs = []

  for target_embedding, target_mask in zip(target_embeddings, target_masks):
    DTW_costs = partial_DTW_torch(input_embedding, target_embedding[target_mask])
    costs.append(DTW_costs)
  costs.append(torch.full((len(input_embedding),), threshold.item(), requires_grad=True, device=input_embedding.device))
  results = F.softmax(-torch.stack(costs), dim=0).detach().cpu().numpy()
  costs = torch.stack(costs).permute((1, 0)).contiguous()
  loss = F.cross_entropy(-costs, labels, ignore_index=-2)

  prevPredict = None
  prevTarget = None
  sentence=[]
  target_sentence=[]
  for i, (label, result) in enumerate(zip(labels, np.argmax(results, axis=0))):
    label = label.item()
    if labels[i] == -2:
      continue

    if prevPredict!=result and results[result][i] > 0.6:
      prevPredict = result
      if result!=len(target_embeddings):
        sentence.append(result)

    if prevTarget!=label:
      prevTarget = label
      if label!=len(target_embeddings):
        target_sentence.append(label)


  distance = min(Levenshtein.distance(''.join(map(str, sentence)), ''.join(map(str, target_sentence))),
                 Levenshtein.distance(''.join(map(str, sentence[1:])), ''.join(map(str, target_sentence))),
                 Levenshtein.distance(''.join(map(str, sentence[:-1])), ''.join(map(str, target_sentence))),
                 Levenshtein.distance(''.join(map(str, sentence[1:-1])), ''.join(map(str, target_sentence))))

  total = len(target_sentence)
  correct = total - distance

  return loss, correct, total



In [ ]:

def train_protonet(model, train_loader, optimizer, device):
    model.train()
    total_loss = 0
    correct_predictions = 0
    total_predictions = 0
    alpha = 0.05
    partition_lambda = 1.0
    for i, batch in enumerate(train_loader):
        input_sequence, input_label, target_sequences, target_labels, target_masks = [b.to(device)[0] for b in batch]
        label_to_index = {label.item(): idx for idx, label in enumerate(target_labels)}
        label_to_index[-1] = len(target_labels)
        label_to_index[-2] = -2
        labels = torch.empty_like(input_label, device=device)
        for i, value in enumerate(input_label):
            labels[i] = label_to_index.get(value.item(), len(target_labels))

        optimizer.zero_grad()
        input_embedding = model(input_sequence.unsqueeze(0)).squeeze(0)
        target_embeddings = [model(target_sequence[target_mask].unsqueeze(0)).squeeze(0) for target_sequence, target_mask in zip(target_sequences, target_masks)]

        loss, partition_correct, partition_total = dtw_partition_loss(input_embedding, target_embeddings, labels, model.threshold, alpha=alpha)

        correct_predictions += partition_correct
        total_predictions += partition_total

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    accuracy = correct_predictions/total_predictions

    return avg_loss, accuracy

def evaluate_protonet(model, val_loader, device):
    model.eval()
    total_loss = 0
    correct_predictions = 0
    total_predictions = 0

    alpha = 0.05
    for i, batch in enumerate(train_loader):
        input_sequence, input_label, target_sequences, target_labels, target_masks = [b.to(device)[0] for b in batch]
        label_to_index = {label.item(): idx for idx, label in enumerate(target_labels)}
        label_to_index[-1] = len(target_labels)
        label_to_index[-2] = -2
        labels = torch.empty_like(input_label, device=device)
        for i, value in enumerate(input_label):
            labels[i] = label_to_index.get(value.item(), len(target_labels))

        input_embedding = model(input_sequence.unsqueeze(0)).squeeze(0)
        target_embeddings = [model(target_sequence[target_mask].unsqueeze(0)).squeeze(0) for target_sequence, target_mask in zip(target_sequences, target_masks)]

        loss, partition_correct, partition_total = dtw_partition_loss(input_embedding, target_embeddings, labels, model.threshold, alpha=alpha)

        total_predictions += partition_total
        correct_predictions += partition_correct

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    accuracy = correct_predictions / total_predictions

    return avg_loss, accuracy


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embedder = TCNSignEmbedding().to(device)
state_dict = torch.load('/content/drive/MyDrive/SignData/protomodel/model-5.h5', map_location=device)
if 'threshold' not in state_dict.keys():
  state_dict['threshold'] = torch.tensor(4.0)
embedder.load_state_dict(state_dict)
optimizer = torch.optim.Adam(embedder.parameters(), lr=0.001)

<ipython-input-7-eb8d5ab2ed6c>:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('/content/drive/MyDrive/SignData/protomodel/model-5.h5', map_location

In [ ]:
num_epochs = 400
# torch.autograd.set_detect_anomaly(True)
for epoch in range(num_epochs):
    train_loss, train_acc =  train_protonet(embedder, train_loader, optimizer, device)
    eval_loss, eval_acc = evaluate_protonet(embedder, val_loader, device)
    torch.save(embedder.state_dict(), f'/content/drive/MyDrive/SignData/protomodel/model-{epoch+1}.h5')

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Eval Loss: {eval_loss:.4f}, Eval Acc: {eval_acc:.4f}")
    print("-----------------------------")




Epoch 1/400
Train Loss: 21.0427, Train Acc: 0.6898
Eval Loss: 22.1828, Eval Acc: 0.5454
-----------------------------
Epoch 2/400
Train Loss: 15.7219, Train Acc: 0.7604
Eval Loss: 22.9395, Eval Acc: 0.5258
-----------------------------
Epoch 3/400
Train Loss: 13.7266, Train Acc: 0.7928
Eval Loss: 20.8415, Eval Acc: 0.5786
-----------------------------
Epoch 4/400
Train Loss: 13.3622, Train Acc: 0.7682
Eval Loss: 17.9881, Eval Acc: 0.6320
-----------------------------
Epoch 5/400
Train Loss: 11.1845, Train Acc: 0.8317
Eval Loss: 19.2448, Eval Acc: 0.5963
-----------------------------
Epoch 6/400
Train Loss: 10.9555, Train Acc: 0.8134
Eval Loss: 20.2850, Eval Acc: 0.5524
-----------------------------
Epoch 7/400
Train Loss: 11.1567, Train Acc: 0.8275
Eval Loss: 19.4714, Eval Acc: 0.5673
-----------------------------
Epoch 8/400
Train Loss: 9.2603, Train Acc: 0.8566
Eval Loss: 17.0657, Eval Acc: 0.5949
-----------------------------


In [ ]:
torch.save(embedder.state_dict(), f'/content/drive/MyDrive/SignData/protomodel/protoModelFinal.h5')

In [ ]:
@njit
def partial_DTW(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    N, E = x.shape
    M, _ = y.shape

    dtw = np.empty((N, M))
    origins = np.empty((N, M))

    for i in range(N):
        dist = np.linalg.norm(x[i] - y[0])
        dtw[i, 0] = dist
        origins[i, 0] = i
    for j in range(1, M):
        dist = np.linalg.norm(x[0] - y[j])
        dtw[0, j] = dist + dtw[0, j-1]
        origins[0, j] = 0

    for i in range(1, N):
        for j in range(1, M):
            dist = np.linalg.norm(x[i] - y[j])

            # Manually determine the minimum cost path
            cost_diag = dtw[i-1, j-1]
            cost_left = dtw[i, j-1]
            cost_up = dtw[i-1, j]

            if cost_diag <= cost_left and cost_diag <= cost_up:
                dtw[i, j] = dist + cost_diag
                origins[i, j] = origins[i-1, j-1]
            elif cost_left < cost_up:
                dtw[i, j] = dist + cost_left
                origins[i, j] = origins[i, j-1]
            else:
                dtw[i, j] = dist + cost_up
                origins[i, j] = origins[i-1, j]

    min_costs = dtw[:, -1]

    curr_min_cost = float('inf')
    curr_origin = origins[-1, -1]
    for i in range(N-1, -1, -1):
        if i < curr_origin:
            curr_min_cost += dtw[i, 0]
        if curr_min_cost > dtw[i, -1]:
            curr_min_cost = dtw[i, -1]
            curr_origin = origins[i, -1]
        min_costs[i] = curr_min_cost

    return min_costs



In [ ]:
input_sequence, input_label, target_sequences, target_labels, target_masks = [b.to(device)[0] for b in next(iter(val_loader))]
index_to_name = {index: name for name, index in train_dataset.class_to_idx.items()}


In [ ]:
input = np.load('/content/drive/MyDrive/SignData/name-73aa5775-5703-4124-b605-5eff592df717.npy')
input_sequence = torch.tensor(input, dtype=torch.float32, device=device)
input_label = torch.full((len(input_sequence),), -1)

In [ ]:
label_to_index = {label.item(): idx for idx, label in enumerate(target_labels)}
label_to_index[-1] = len(target_labels)
labels = torch.empty_like(input_label, device=device)
for i, value in enumerate(input_label):
    labels[i] = label_to_index.get(value.item(), len(target_labels))

input_embedding = embedder(input_sequence.unsqueeze(0)).squeeze(0)
target_embeddings = embedder(target_sequences)

partition_loss, correct, total = dtw_partition_loss(input_embedding, target_embeddings, target_masks, labels, embedder.threshold)

print(correct, total)

In [ ]:
labels

In [ ]:
prev=None
groundTruth=[]
startEnd={}
for i, label in enumerate(labels):
  label=label.item()
  if label!=prev:
    if prev in startEnd:
      startEnd[prev].append(i-1)
    prev=label
    if label==len(target_embeddings):
      continue
    startEnd[label]=[i]
    groundTruth.append(label)


print(groundTruth)
print(startEnd)

In [ ]:
y_vals = []

for target_embedding, target_mask in zip(target_embeddings, target_masks):
  costs = partial_DTW(input_embedding.detach().cpu().numpy(), target_embedding[target_mask].detach().cpu().numpy())
  y_vals.append(-costs/len(target_embedding[target_mask]))
y_vals.append(torch.full((len(y_vals[0]),), -0.5))
y_vals = np.array(y_vals)
# y_vals = F.softmax(torch.tensor(y_vals), dim=0).detach().cpu().numpy()
x_values = np.arange(y_vals.shape[1])


In [ ]:
index_to_name[np.argmax(y_vals, axis=0)[0]]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

index_to_name[-1] = "None"

colors = list(cm.rainbow(np.linspace(0, 1, len(target_embeddings)))) + ['black']
# visible = groundTruth + [len(target_embeddings)]
visible = [val_dataset.class_to_idx[x] for x in ['name']]
for i in range(y_vals.shape[0]):
    # if not i in visible:
    #   continue
    if i in visible:
      colors[i] = 'black'
    plt.plot(x_values, y_vals[i], label=index_to_name[target_labels[i].item() if i!=len(target_embeddings) else -1], color=colors[i])
    # plt.plot(x_values, y_vals[i], label=index_to_name[target_labels[i].item() if i!=len(target_embeddings) else -1])

# Optionally, add labels and a legend
plt.ylim(-0.5, 0)
plt.xlabel('X')
plt.ylabel('Y')
plt.title('N Lines Plot')
for num in range(len(target_embeddings)):
  if num in startEnd:
    start, end = startEnd[num]
    plt.axvspan(start, end, color=colors[num], alpha=0.5, label=index_to_name[target_labels[num].item() if num!=len(target_embeddings) else -1])

plt.legend(loc='upper left', bbox_to_anchor=(1, 1))

# Show the plot
print(groundTruth)
plt.show()
